# Vision fine-tuning for dog breed classification

This executable demo uses `DefaultAzureCredential` with the specified Azure AI
Foundry project, the supported `gpt-4o-2024-08-06` vision fine-tuning model,
and a bounded four-breed Stanford Dogs subset.

In [1]:
from pathlib import Path
import json
import pandas as pd
from sklearn.metrics import accuracy_score

from demo_common import (
    ACCOUNT_NAME, BASE_DEPLOYMENT, BASE_MODEL, FT_DEPLOYMENT, PROJECT_ENDPOINT,
    SUCCESSFUL_JOB_ID,
    build_demo_dataset, classification_prompt, classify, deploy_fine_tuned_model,
    download_dataset, get_openai_client, image_data_uri, normalize_prediction,
    project_job_link, wait_for_file, wait_for_job, write_fine_tuning_jsonl,
)

ROOT = Path.cwd()
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)
client = get_openai_client()
print("Project:", PROJECT_ENDPOINT)
print("Base model:", BASE_MODEL)
print("Base deployment:", BASE_DEPLOYMENT)

Project: https://eastus2-prakharg-demo-2026.services.ai.azure.com/api/projects/eastus2-prakharg-demo-2026
Base model: gpt-4o-2024-08-06
Base deployment: image-breed-gpt-4o-base


In [2]:
images_root = download_dataset(ROOT / "data")
dataset_df = build_demo_dataset(images_root)
dataset_df.to_csv(OUTPUTS / "dataset_manifest.csv", index=False)
print(dataset_df.groupby(["split", "breed"]).size().unstack(fill_value=0))

breed       Chihuahua  Newfoundland  Saint Bernard  Weimaraner
split                                                         
test                4             4              4           4
train              12            12             12          12
validation          4             4              4           4


In [3]:
labels = sorted(dataset_df["breed"].unique())
prompt = classification_prompt(labels)
test_df = dataset_df[dataset_df["split"] == "test"].copy()

base_predictions = []
for row in test_df.itertuples(index=False):
    raw = classify(client, BASE_DEPLOYMENT, prompt, image_data_uri(Path(row.image_path)))
    base_predictions.append(normalize_prediction(raw, labels))
    print(row.id, row.breed, "=>", base_predictions[-1])

test_df["base_prediction"] = base_predictions
base_accuracy = accuracy_score(test_df["breed"], test_df["base_prediction"])
print(f"Base accuracy: {base_accuracy:.3f}")

n02085620-Chihuahua-16 Chihuahua => Chihuahua


n02085620-Chihuahua-17 Chihuahua => Chihuahua


n02085620-Chihuahua-18 Chihuahua => Chihuahua


n02085620-Chihuahua-19 Chihuahua => Chihuahua


n02092339-Weimaraner-16 Weimaraner => Weimaraner


...

n02109525-Saint_Bernard-19 Saint Bernard => Saint Bernard


n02111277-Newfoundland-16 Newfoundland => Newfoundland


n02111277-Newfoundland-17 Newfoundland => Newfoundland


n02111277-Newfoundland-18 Newfoundland => Newfoundland


n02111277-Newfoundland-19 Newfoundland => Newfoundland
Base accuracy: 0.938


In [4]:
train_file = OUTPUTS / "train_classification.jsonl"
validation_file = OUTPUTS / "validation_classification.jsonl"
write_fine_tuning_jsonl(
    dataset_df[dataset_df["split"] == "train"], train_file, prompt
)
write_fine_tuning_jsonl(
    dataset_df[dataset_df["split"] == "validation"], validation_file, prompt
)
print("Training examples:", sum(1 for _ in train_file.open(encoding="utf-8")))
print("Validation examples:", sum(1 for _ in validation_file.open(encoding="utf-8")))

Training examples: 48
Validation examples: 16


In [5]:
training_upload = client.files.create(file=train_file.open("rb"), purpose="fine-tune")
validation_upload = client.files.create(
    file=validation_file.open("rb"), purpose="fine-tune"
)
print("Training file:", training_upload.id)
print("Validation file:", validation_upload.id)
wait_for_file(client, training_upload.id)
wait_for_file(client, validation_upload.id)

job = client.fine_tuning.jobs.retrieve(SUCCESSFUL_JOB_ID)
print("Using completed job:", job.id)
print("Job link:", project_job_link(job.id))
print(job.model_dump_json(indent=2))
if job.status != "succeeded":
    raise RuntimeError(f"Fine-tuning job {job.id} ended as {job.status}: {job.error}")

Training file: file-b966d33b2ed34e89beefe2b066c2daa3
Validation file: file-ec1129be617d46c6826ed9cbaec2164d


File file-b966d33b2ed34e89beefe2b066c2daa3: processed


File file-ec1129be617d46c6826ed9cbaec2164d: pending


File file-ec1129be617d46c6826ed9cbaec2164d: processed


Using completed job: ftjob-25d544adc35d47e1bbcf92b3977037d3
Job link: https://ai.azure.com/fine-tuning/ftjob-25d544adc35d47e1bbcf92b3977037d3?wsid=/subscriptions/ba7979f7-d040-49c9-af1a-7414402bf622/resourceGroups/prakharg-demo-2026/providers/Microsoft.CognitiveServices/accounts/eastus2-prakharg-demo-2026/projects/eastus2-prakharg-demo-2026
{
  "id": "ftjob-25d544adc35d47e1bbcf92b3977037d3",
  "created_at": 1791038738,
  "error": null,
  "fine_tuned_model": "gpt-4o-2024-08-06.ft-25d544adc35d47e1bbcf92b3977037d3-image-breed-demo",
  "finished_at": 1791043342,
  "hyperparameters": {
    "batch_size": 1,
    "learning_rate_multiplier": 1.0,
    "n_epochs": 1
  },
  "model": "gpt-4o-2024-08-06",
  "object": "fine_tuning.job",
  "organization_id": null,
  "result_files": [
    "file-db15069a90c1412ca41d81924839d680"
  ],
  "seed": 42,
  "status": "succeeded",
  "trained_tokens": 379793,
  "training_file": "file-6cbdd927f4e2439abcf0635123272b49",
  "validation_file": "file-49c1e4b7fd6d4bd8a0

In [6]:
deployment = deploy_fine_tuned_model(job.fine_tuned_model)
print("Fine-tuned deployment:", deployment["name"])
print("Provisioning state:", deployment["properties"]["provisioningState"])

Deployment image-breed-gpt-4o-ft: Creating


Deployment image-breed-gpt-4o-ft: Creating


Deployment image-breed-gpt-4o-ft: Creating


Deployment image-breed-gpt-4o-ft: Succeeded
Fine-tuned deployment: image-breed-gpt-4o-ft
Provisioning state: Succeeded


In [7]:
ft_predictions = []
for row in test_df.itertuples(index=False):
    raw = classify(client, FT_DEPLOYMENT, prompt, image_data_uri(Path(row.image_path)))
    ft_predictions.append(normalize_prediction(raw, labels))
    print(row.id, row.breed, "=>", ft_predictions[-1])

test_df["ft_prediction"] = ft_predictions
ft_accuracy = accuracy_score(test_df["breed"], test_df["ft_prediction"])
accuracy_delta = ft_accuracy - base_accuracy
print(f"Fine-tuned accuracy: {ft_accuracy:.3f}")
print(f"Accuracy delta: {accuracy_delta:+.3f}")

test_df.to_csv(OUTPUTS / "accuracy_predictions.csv", index=False)
run_summary = {
    "project_endpoint": PROJECT_ENDPOINT,
    "resource": ACCOUNT_NAME,
    "base_model": BASE_MODEL,
    "base_deployment": BASE_DEPLOYMENT,
    "fine_tuned_model": job.fine_tuned_model,
    "fine_tuned_deployment": FT_DEPLOYMENT,
    "training_file_id": training_upload.id,
    "validation_file_id": validation_upload.id,
    "job_id": job.id,
    "job_link": project_job_link(job.id),
    "job_status": job.status,
    "base_accuracy": base_accuracy,
    "fine_tuned_accuracy": ft_accuracy,
    "accuracy_delta": accuracy_delta,
    "test_examples": len(test_df),
}
(OUTPUTS / "run_summary.json").write_text(
    json.dumps(run_summary, indent=2), encoding="utf-8"
)
run_summary

n02085620-Chihuahua-16 Chihuahua => Chihuahua


n02085620-Chihuahua-17 Chihuahua => Chihuahua


n02085620-Chihuahua-18 Chihuahua => Chihuahua


n02085620-Chihuahua-19 Chihuahua => Chihuahua


n02092339-Weimaraner-16 Weimaraner => Weimaraner


...

n02111277-Newfoundland-16 Newfoundland => Newfoundland


n02111277-Newfoundland-17 Newfoundland => Newfoundland


n02111277-Newfoundland-18 Newfoundland => Newfoundland


n02111277-Newfoundland-19 Newfoundland => Newfoundland
Fine-tuned accuracy: 0.938
Accuracy delta: +0.000


{'project_endpoint': 'https://eastus2-prakharg-demo-2026.services.ai.azure.com/api/projects/eastus2-prakharg-demo-2026',
 'resource': 'eastus2-prakharg-demo-2026',
 'base_model': 'gpt-4o-2024-08-06',
 'base_deployment': 'image-breed-gpt-4o-base',
 'fine_tuned_model': 'gpt-4o-2024-08-06.ft-25d544adc35d47e1bbcf92b3977037d3-image-breed-demo',
 'fine_tuned_deployment': 'image-breed-gpt-4o-ft',
 'training_file_id': 'file-b966d33b2ed34e89beefe2b066c2daa3',
 'validation_file_id': 'file-ec1129be617d46c6826ed9cbaec2164d',
 'job_id': 'ftjob-25d544adc35d47e1bbcf92b3977037d3',
 'job_link': 'https://ai.azure.com/fine-tuning/ftjob-25d544adc35d47e1bbcf92b3977037d3?wsid=/subscriptions/ba7979f7-d040-49c9-af1a-7414402bf622/resourceGroups/prakharg-demo-2026/providers/Microsoft.CognitiveServices/accounts/eastus2-prakharg-demo-2026/projects/eastus2-prakharg-demo-2026',
 'job_status': 'succeeded',
 'base_accuracy': 0.9375,
 'fine_tuned_accuracy': 0.9375,
 'accuracy_delta': 0.0,
 'test_examples': 16}